# Composition Explorer — Colab Edition

数学と美術理論で、まだ見ぬ構図を発見する。

このノートブックは GitHub Pages 版と**同じ UI** を Colab 上で起動し、リンクから開けるようにします。
UI は自動的に Colab バックエンドを検出し（設定ページに表示）、今後 Colab の計算資源（GPU など）を使う機能が追加されていきます。

1. 下のセルを順に実行します（GPU は「ランタイム → ランタイムのタイプを変更」で有効化できます）。
2. 最後のセルに表示されるリンクを開きます。

Repository: https://github.com/MasakiNiwa/Composition-Explorer (MIT License)

In [ ]:
#@title 1. 準備（サーバーと UI の取得） { display-mode: "form" }
#@markdown `prebuilt`: GitHub Pages で公開中のビルドをダウンロード（高速）／ `source`: 指定ブランチからビルド（Node.js が必要）
SOURCE = "prebuilt"  #@param ["prebuilt", "source"]
BRANCH = "main"  #@param {type:"string"}

import os, subprocess, zipfile, urllib.request, shutil

REPO = "https://github.com/MasakiNiwa/Composition-Explorer.git"
PAGES = "https://masakiniwa.github.io/Composition-Explorer/"
WORK = "/content/composition-explorer"

if os.path.exists(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, WORK], check=True)

SITE = os.path.join(WORK, "site")
if SOURCE == "prebuilt":
    zpath = os.path.join(WORK, "site.zip")
    urllib.request.urlretrieve(PAGES + "site.zip", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(SITE)
else:
    node = subprocess.run(["bash", "-lc", "node -v || true"], capture_output=True, text=True).stdout.strip()
    major = int(node.lstrip("v").split(".")[0]) if node.startswith("v") else 0
    if major < 22:
        subprocess.run(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs"], check=True)
    subprocess.run(["bash", "-lc", f"cd {WORK} && npm ci && npm run build"], check=True)
    shutil.copytree(os.path.join(WORK, "dist"), SITE)
print("UI ready:", SITE)

In [ ]:
#@title 2. サーバー起動とリンク表示 { display-mode: "form" }
import sys
sys.path.insert(0, os.path.join(WORK, "colab"))
import importlib, server
importlib.reload(server)

PORT = 8000
try:
    httpd.shutdown()  # restart if this cell is run again
except NameError:
    pass
httpd = server.start(root=SITE, port=PORT)

from google.colab.output import eval_js
from IPython.display import HTML, display

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
gpu = server.gpu_info()
display(HTML(f'''
<p style="font-size:16px">🎨 <a href="{url}" target="_blank"><b>Composition Explorer を開く</b></a></p>
<p>GPU: {gpu[0]["name"] if gpu else "なし（CPU ランタイム）"}</p>
'''))

## メモ

- UI は GitHub Pages 版と同一のビルドです。保存データはブラウザ（このプロキシのドメイン）に保存されます。
- サーバーは `colab/server.py`（標準ライブラリのみ）。`api/health`・`api/info` を提供し、
  今後 GPU を使う評価・画像生成などのエンドポイントを追加していきます。